# Entraînement de l'IA Malaria (YOLOv8)

Ce notebook permet d'entraîner le modèle médical sur les serveurs Google avec carte graphique GPU, réduisant le temps de 15 jours à environ 2 ou 3 heures.

In [ ]:
!pip install ultralytics pyyaml

In [ ]:
# Connecter Google Drive pour récupérer le dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Dézipper le dataset depuis Google Drive vers l'espace de travail hyper-rapide de Colab
!cp "/content/drive/MyDrive/nlm_pf_yolo.zip" /content/
!unzip -q /content/nlm_pf_yolo.zip -d /content/
print("Dataset dézippé !")

In [ ]:
import yaml

# Mettre à jour le chemin dans dataset.yaml pour qu'il trouve les images sur Colab
yaml_path = '/content/nlm_pf_yolo/dataset.yaml'
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = '/content/nlm_pf_yolo'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, sort_keys=False)

print("Chemin mis à jour dans dataset.yaml")

In [ ]:
from ultralytics import YOLO

# Charger le modèle Nano de base (très léger pour le mobile)
model = YOLO('yolov8n.pt')

# Lancer l'entraînement avec le GPU (device=0)
results = model.train(
    data='/content/nlm_pf_yolo/dataset.yaml',
    epochs=100,           # Nombre de cycles
    imgsz=640,            # Taille des images
    batch=16,             # Nombre d'images par itération
    project='malaria_detect',
    name='yolov8n_run',
    patience=20,          # Arrêt prématuré si l'IA stagne
    device=0              # Force l'utilisation du GPU Nvidia de Google
)

In [ ]:
# Exporter en TFLite pour l'application Flutter
import shutil
import os

print("Exportation du modèle pour le mobile...")
model.export(format='tflite')

print("✅ Modèle exporté !")

In [ ]:
# Copier le modèle final vers Google Drive pour que tu puisses le télécharger
import os
import shutil

tflite_dir = "/content/malaria_detect/yolov8n_run/weights/best_saved_model/"
tflite_file = "best_float32.tflite"

if not os.path.exists(os.path.join(tflite_dir, tflite_file)):
    tflite_file = "best_float16.tflite"

if os.path.exists(os.path.join(tflite_dir, tflite_file)):
    shutil.copy(os.path.join(tflite_dir, tflite_file), "/content/drive/MyDrive/malaria_yolov8n.tflite")
    print("🎉 Succès ! Le fichier malaria_yolov8n.tflite a été copié à la racine de ton Google Drive.")
    print("Tu n'as plus qu'à le télécharger et le placer dans le dossier 'assets/' de ton projet Flutter.")
else:
    print("⚠️ Fichier TFLite introuvable. Vérifie le dossier ", tflite_dir)